In [1]:
import json
import pandas as pd
import os
from openai import OpenAI
from dotenv import load_dotenv

In [2]:
# ---------------------------------------
# 1. Initialize OpenAI client
# ---------------------------------------
load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY")

if not API_KEY:
    raise SystemExit("OPENAI_API_KEY is not set. Add it to your .env file.")

client = OpenAI(api_key=API_KEY)


In [3]:
# ---------------------------------------
# 2. Load CSV data
# ---------------------------------------

df = pd.read_csv("data/transcriptions.csv")

print("Number of records:", len(df))
print(df.head())

Number of records: 54
  medical_specialty                                      transcription
0        Cardiology  A 65-year-old male presents with chest pain on...
1        Cardiology  A 58-year-old woman reports intermittent palpi...
2        Cardiology  A 72-year-old male has shortness of breath and...
3        Cardiology  A 49-year-old patient has elevated cholesterol...
4        Cardiology  A 61-year-old woman has persistent hypertensio...


In [ ]:
# ---------------------------------------
# 3. Function to process one transcription
# ---------------------------------------
def extract_medical_data(transcription, medical_specialty):
    """
    Extract age, treatment/procedure and ICD code
    from a medical transcription using one OpenAI call.
    """

    messages = [
        {
            "role": "system",
            "content": """
You are a healthcare data extraction assistant.

Extract structured information from the medical transcription.

Return:
1. Patient age
2. Recommended treatment or procedure
3. ICD-10-CM code related to the recommended treatment,
   procedure, or medical condition.

If information is missing, return "Unknown".

The ICD code should be treated as an AI-generated suggestion
and should be verified by a qualified medical coding professional.
"""
        },
        {
            "role": "user",
            "content": f"""
Medical Specialty:
{medical_specialty}

Medical Transcription:
{transcription}

Extract the requested medical information.
"""
        }
    ]


In [ ]:
# ---------------------------------------
# Function definition
# ---------------------------------------
    tools = [
        {
            "type": "function",
            "function": {
                "name": "extract_medical_data",
                "description": "Extract structured medical information",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "Age": {
                            "type": "integer",
                            "description": "Patient's age"
                        },
                        "Recommended Treatment/Procedure": {
                            "type": "string",
                            "description": (
                                "Recommended treatment or medical procedure"
                            )
                        },
                        "ICD Code": {
                            "type": "string",
                            "description": (
                                "Suggested ICD-10-CM code related "
                                "to the condition, treatment, or procedure"
                            )
                        }
                    },
                    "required": [
                        "Age",
                        "Recommended Treatment/Procedure",
                        "ICD Code"
                    ]
                }
            }
        }
    ]

In [ ]:
# ---------------------------------------
# ONE OpenAI API CALL
# ---------------------------------------
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=tools,
        tool_choice={
            "type": "function",
            "function": {
                "name": "extract_medical_data"
            }
        }
    )

NameError: name 'messages' is not defined

In [ ]:
tool_call = response.choices[0].message.tool_calls[0]

arguments = tool_call.function.arguments

return json.loads(arguments)